# Probing pipeline
Runs data creation, training, evaluation, visualisation and control experiments end to end.

## Imports

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from transformer_lens import HookedTransformer

from canonical.probing.config import RunConfig
from canonical.probing.utils import (
    build_classifiers,
    create_results_path,
    save_clf_with_skops,
    split_by_layer,
)
from canonical.probing.probes_dataset_creation_script import (
    create_canonical_dataset,
    distractor_word_data,
    neutral_filler_data,
    no_rule_keyword,
    opposite_statuses_rules,
)
from canonical.probing.train_probes import training
from canonical.probing.evaluate_probes import evaluate
from canonical.probing.plot_probes import (
    accuracies_from_evaluation_results,
    plot_accuracy_per_layer,
    plot_auroc_curves,
)
from canonical.probing.control_experiments import (
    confound_datasets_control,
    p_value_control,
    train_on_shuffled_labels,
    weights_vs_diff_of_means,
)


## Log in to HuggingFace

In [ ]:
load_dotenv()
login(token=os.environ["HF_TOKEN"])


## Config

### English


In [ ]:
BASE_DIR = "/workspace/crosslingual-rule-following"  # adjust if the repo lives elsewhere on the pod
PROBES_DATA_DIR = f"{BASE_DIR}/canonical/probing/probes_data"

LANGUAGE = "en"
DATASET_NAME = f"rule-following-eval_{LANGUAGE}"
RESULTS_FOLDER = f"{BASE_DIR}/canonical/probing/results_{LANGUAGE}"

HF_REPO_IX = "crosslingual-rule-following/canonical-dataset"
HF_REPO_TYPE = "dataset"
#ACTIVATIONS_IN_HF = ""
#Y_IN_HF = ""
#TEXT_INDEX_IN_HF = ""
#ORIGINAL_TEXT_HF = ""
#JSONL_IN_HF = "no_keyword_source.jsonl"

MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_NAME_LLAMA = "meta-llama/Llama-3.1-8B-Instruct"
HOOK_NAME = "hook_resid_post"
POS_SLICE = -1

CLASSIFIER_SPEC = {"logistic_regression": {"max_iter": 2000}, "mlp": {}, "knn": {}}
N_PERM = 1000


## Load the model
Used for on-the-fly activation extraction and to read off the layer count.

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME)
#model2 = HookedTransformer.from_pretrained(MODEL_NAME_LLAMA)
n_layers = model.cfg.n_layers
#n_layers2 = model2.cfg.n_layers

## Run config

In [ ]:
run_cfg = RunConfig(
    language=LANGUAGE,
    n_layers=n_layers,
    dataset_name=DATASET_NAME,
    results_folder=RESULTS_FOLDER,
)
create_results_path(run_cfg)


## Classifiers

In [ ]:
classifiers = build_classifiers(CLASSIFIER_SPEC)


## Data creation
Main train/test/held-out split, pulled from HuggingFace.

In [ ]:
json_data_path = "data/en/test.jsonl"

In [ ]:
dataset = create_canonical_dataset(
    jsonl_in_hf=json_data_path,
    hf_repo_ix=HF_REPO_IX,
    hf_repo_type=HF_REPO_TYPE,
    model=model,
    hook_name=HOOK_NAME,
    pos_slice=POS_SLICE,
)


Confound datasets, for later control experiments.

In [ ]:
neutral_filler_dataset = neutral_filler_data(
    f"{PROBES_DATA_DIR}/neutral_fillers.json", model, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)


In [ ]:
distractor_dataset = distractor_word_data(
    json_data_path, HF_REPO_IX, model, hf_repo_type=HF_REPO_TYPE
)


In [ ]:
no_keyword_dataset = no_rule_keyword(
    model, json_data_path, HF_REPO_IX, repo_type=HF_REPO_TYPE, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)


In [ ]:
double_rule_dataset = opposite_statuses_rules(
    f"{PROBES_DATA_DIR}/double_rule_dataset.json", model, hook_name=HOOK_NAME, pos_slice=POS_SLICE
)


## Training

In [ ]:
train_X = split_by_layer(dataset.train_x) # util to split tensor into 2D for clf training
trained_classifiers = training(run_cfg, classifiers, train_X, dataset.train_y)


In [ ]:
trained_probes_path = save_clf_with_skops(run_cfg, trained_classifiers)


## Evaluation

In [ ]:
test_X = split_by_layer(dataset.test_x)
validation_evals, validation_eval_path = evaluate(
    run_cfg, trained_classifiers, test_X, dataset.test_y, save_path_prefix="Valid"
)


In [ ]:
held_X = split_by_layer(dataset.held_x)
held_out_evals, held_eval_path = evaluate(
    run_cfg, trained_classifiers, held_X, dataset.held_y, save_path_prefix="Held"
)


## Visualisation

In [ ]:
validation_accuracies, validation_legend = accuracies_from_evaluation_results(validation_evals)
plot_accuracy_per_layer(run_cfg, validation_accuracies, validation_legend, save_path_prefix="Valid")


In [ ]:
held_accuracies, held_legend = accuracies_from_evaluation_results(held_out_evals)
plot_accuracy_per_layer(run_cfg, held_accuracies, held_legend, save_path_prefix="Held")


AUROC curves, one per classifier/layer, on the held-out set.

In [ ]:
y_trues, y_scores, legend = [], [], []
for name, layer_clfs in trained_classifiers.items():
    for layer, clf in layer_clfs.items():
        y_trues.append(dataset.held_y)
        y_scores.append(clf.predict_proba(held_X[layer])[:, 1])
        legend.append(f"{name} layer {layer}")


In [ ]:
plot_auroc_curves(run_cfg, y_trues, y_scores, legend, save_path_prefix="Held")


## Control experiments

In [ ]:
shuffled_results = train_on_shuffled_labels(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
)


In [ ]:
p_value_results = p_value_control(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    n_perm=N_PERM,
    load_normal_eval_scores=f"{run_cfg.eval_path}/ValidEval_{run_cfg.language}.json",
)


In [ ]:
weights_results = weights_vs_diff_of_means(
    run_cfg, json_data_path, HF_REPO_IX, classifiers,
    hf_repo_type=HF_REPO_TYPE, model=model, hook_name=HOOK_NAME, pos_slice=POS_SLICE,
    trained_clfs_folder=trained_probes_path,
)


In [ ]:
confound_results = confound_datasets_control(
    run_cfg, json_data_path, HF_REPO_IX, classifiers, model,
    f"{PROBES_DATA_DIR}/neutral_fillers.json",
    json_data_path,
    json_data_path,
    f"{PROBES_DATA_DIR}/double_rule_dataset.json",
    hf_repo_type=HF_REPO_TYPE,
    trained_clfs_folder=trained_probes_path,
    n_perm=N_PERM,
)


## Canonical vs confound: accuracy and AUROC
Compares held-out canonical performance against each confound dataset, per classifier.

In [ ]:
from sklearn.metrics import classification_report

confound_datasets_by_name = {
    "NeutralFiller": (neutral_filler_dataset.neutral_x, neutral_filler_dataset.neutral_y),
    "Distractor": (distractor_dataset.distractor_x, distractor_dataset.distractor_y),
    "NoKeyword": (no_keyword_dataset.nokrule_x, no_keyword_dataset.nokrule_y),
    "DoubleRule": (double_rule_dataset.doublerule_x, double_rule_dataset.doublerule_y),
}


In [ ]:
canonical_accuracies, canonical_legend = accuracies_from_evaluation_results(held_out_evals)
canonical_legend = [f"Canonical-{name}" for name in canonical_legend]

confound_accuracies, confound_legend = [], []
for confound_name, (confound_x, confound_y) in confound_datasets_by_name.items():
    confound_X = split_by_layer(confound_x)
    confound_y = np.array(confound_y)
    for name, layer_clfs in trained_classifiers.items():
        layer_accuracies = {}
        for layer, clf in layer_clfs.items():
            predictions = clf.predict(confound_X[layer])
            layer_accuracies[layer] = classification_report(confound_y, predictions, output_dict=True)["accuracy"]
        confound_accuracies.append(layer_accuracies)
        confound_legend.append(f"{confound_name}-{name}")


In [ ]:
plot_accuracy_per_layer(
    run_cfg, canonical_accuracies + confound_accuracies, canonical_legend + confound_legend,
    save_path_prefix="CanonicalVsConfound",
)


AUROC at each classifier's best canonical held-out layer (all layers x all datasets would be unreadable).

In [ ]:
best_layer_per_classifier = {
    name: max(layer_dict, key=lambda l: layer_dict[l]["accuracy"])
    for name, layer_dict in held_out_evals.items()
}


In [ ]:
canonical_confound_y_trues, canonical_confound_y_scores, canonical_confound_legend = [], [], []
for name, layer_clfs in trained_classifiers.items():
    best_layer = best_layer_per_classifier[name]
    clf = layer_clfs[best_layer]
    canonical_confound_y_trues.append(dataset.held_y)
    canonical_confound_y_scores.append(clf.predict_proba(held_X[best_layer])[:, 1])
    canonical_confound_legend.append(f"Canonical-{name} layer {best_layer}")
    for confound_name, (confound_x, confound_y) in confound_datasets_by_name.items():
        confound_X = split_by_layer(confound_x)
        canonical_confound_y_trues.append(np.array(confound_y))
        canonical_confound_y_scores.append(clf.predict_proba(confound_X[best_layer])[:, 1])
        canonical_confound_legend.append(f"{confound_name}-{name} layer {best_layer}")


In [ ]:
plot_auroc_curves(
    run_cfg, canonical_confound_y_trues, canonical_confound_y_scores, canonical_confound_legend,
    save_path_prefix="CanonicalVsConfound",
)
